# Face Recognition Demo with User Upload

This notebook lets you upload your own face image, compares it against a small sample of LFW faces, and shows which faces are most similar.

In [ ]:
!pip install deepface scikit-learn matplotlib opencv-python-headless --quiet

In [ ]:
from deepface import DeepFace
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people
from google.colab import files
from PIL import Image
import io


In [ ]:
# Load LFW data
min_faces_per_person = 40
lfw_people = fetch_lfw_people(min_faces_per_person=min_faces_per_person, resize=0.4)
print("Number of images in the dataset:", lfw_people.data.shape[0])
print("Number of people in the dataset:", len(np.unique(lfw_people.target)))


In [ ]:
def ensure_rgb(image):
    """Convert grayscale to RGB when needed. DeepFace expects 3-channel arrays."""
    if len(image.shape) == 2:
        image = np.repeat(image[:, :, None], 3, axis=2)
    return image

def load_uploaded_image():
    uploaded = files.upload()
    filename = next(iter(uploaded))
    img_bytes = uploaded[filename]
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img_array = np.array(img)
    return img_array, filename


In [ ]:
uploaded_face, uploaded_filename = load_uploaded_image()
print(f"Uploaded file: {uploaded_filename}")
print("Uploaded image shape:", uploaded_face.shape)
plt.imshow(uploaded_face)
plt.title("Uploaded Face")
plt.axis("off")
plt.show()


In [ ]:
# Pick a target identity from the LFW set for comparison against the uploaded face
target_name = "Arnold Schwarzenegger"
target_index = np.where(lfw_people.target_names == target_name)[0][0]
target_indices = np.where(lfw_people.target == target_index)[0]
num_target_images = 3
target_faces = np.stack([ensure_rgb(face) for face in lfw_people.images[target_indices[:num_target_images]]])

# Include some other people for comparison as well
other_indices = np.where(lfw_people.target != target_index)[0]
other_faces = np.stack([ensure_rgb(face) for face in lfw_people.images[other_indices[:num_target_images]]])

joined_data = [(face, target_name) for face in target_faces]
other_labels = [lfw_people.target_names[lfw_people.target[idx]] for idx in other_indices[:num_target_images]]
joined_data += list(zip(other_faces, other_labels))
np.random.shuffle(joined_data)
joined_faces = np.stack([item[0] for item in joined_data])
joined_labels = [item[1] for item in joined_data]


In [ ]:
# Compute DeepFace embeddings for the uploaded face and all comparison faces
uploaded_embedding = DeepFace.represent(
    ensure_rgb(uploaded_face),
    model_name="VGG-Face",
    enforce_detection=False,
)[0]["embedding"]

comparison_embeddings = [
    DeepFace.represent(face, model_name="VGG-Face", enforce_detection=False)[0]["embedding"]
    for face in joined_faces
]

results = []
for embedding in comparison_embeddings:
    # Cosine similarity compares the direction of the embeddings.
    # Values near 1.0 mean the uploaded face looks similar to that person,
    # while values near 0 are less similar.
    score = np.dot(uploaded_embedding, embedding) / (
        np.linalg.norm(uploaded_embedding) * np.linalg.norm(embedding)
    )
    results.append(score)

print("Comparison scores:", results)


In [ ]:
# Display the uploaded face and the comparison set
num_images = len(joined_faces) + 1
plt.figure(figsize=(18, 6))

ax = plt.subplot(1, num_images, 1)
ax.imshow(uploaded_face)
ax.set_title("Uploaded Face")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("Your image", labelpad=10)

for i in range(len(joined_faces)):
    ax = plt.subplot(1, num_images, i + 2)
    ax.imshow(joined_faces[i], cmap='gray')
    ax.set_title(f"Similarity: {results[i]:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel(joined_labels[i], labelpad=10)

plt.tight_layout()
plt.show()
